In [4]:
import cutlass 
import cutlass.cute as cute 

A (flat) layout, $L$ is a pair of tuples $S:D$ where $S = \text{Shape}(L)$ and $D = \text{Stride}(L)$ of the same length $m = \text{Rank}(L)$ where $S \in \text{Tuple}(\mathbb{N}_{>0})$ and $D \in \text{Tuple}(\mathbb{N})$. The size of the layout $\text{Size}(L) = \prod_{i=1}^m s_{i}$ is the size of its shape, and the co-size of the layout $\text{coSize(L)} = 1 + \sum_{i=1}^m(s_{i}-1)d_{i}$. 

<!-- https://q.uiver.app/#q=WzAsNSxbMCwwLCJbMCwgXFx0ZXh0e1NpemV9KEwpKSIsWzIzNiw2OCw4NiwxXV0sWzIsMCwiWzAsUykiLFsyMzYsNjgsODYsMV1dLFs0LDAsIlswLE4pIixbMjM2LDY4LDg2LDFdXSxbMywxLCIgICIsWzIzNiw2OCw4NiwxXV0sWzUsMCwiXFxtYXRoYmIgTiIsWzIzNiw2OCw4NiwxXV0sWzAsMSwiXFx0ZXh0e2NvbGV4fV57LTF9X1MiLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMSwwLCJcXHRleHR7Y29sZXh9X1MiLDAseyJvZmZzZXQiOi0zLCJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMSwyLCJcXHZhcnBoaV9EXk4iLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMiw0LCIiLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdLCJzdHlsZSI6eyJ0YWlsIjp7Im5hbWUiOiJob29rIiwic2lkZSI6InRvcCJ9fX1dLFsxLDQsIlxcdmFycGhpX0QgIiwwLHsib2Zmc2V0IjoxLCJjdXJ2ZSI6MiwiY29sb3VyIjpbMjM2LDY4LDg2XX0sWzIzNiw2OCw4NiwxXV1d --> <iframe class="quiver-embed" src="https://q.uiver.app/#q=WzAsNSxbMCwwLCJbMCwgXFx0ZXh0e1NpemV9KEwpKSIsWzIzNiw2OCw4NiwxXV0sWzIsMCwiWzAsUykiLFsyMzYsNjgsODYsMV1dLFs0LDAsIlswLE4pIixbMjM2LDY4LDg2LDFdXSxbMywxLCIgICIsWzIzNiw2OCw4NiwxXV0sWzUsMCwiXFxtYXRoYmIgTiIsWzIzNiw2OCw4NiwxXV0sWzAsMSwiXFx0ZXh0e2NvbGV4fV57LTF9X1MiLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMSwwLCJcXHRleHR7Y29sZXh9X1MiLDAseyJvZmZzZXQiOi0zLCJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMSwyLCJcXHZhcnBoaV9EXk4iLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdfSxbMjM2LDY4LDg2LDFdXSxbMiw0LCIiLDAseyJjb2xvdXIiOlsyMzYsNjgsODZdLCJzdHlsZSI6eyJ0YWlsIjp7Im5hbWUiOiJob29rIiwic2lkZSI6InRvcCJ9fX1dLFsxLDQsIlxcdmFycGhpX0QgIiwwLHsib2Zmc2V0IjoxLCJjdXJ2ZSI6MiwiY29sb3VyIjpbMjM2LDY4LDg2XX0sWzIzNiw2OCw4NiwxXV1d&embed" width="700" height="230" style="border-radius: 8px; border: none;"></iframe>
The $\text{colex}_{S}$ map is the isomorphism $$\ket{x_{i}}_{i=1}^m \mapsto \sum_{i=1}^m (\prod_{j=1}^{i-1}s_{j})x_{i}$$
and its inverse is the map $$ x \mapsto \ket{\left\lfloor  \frac{x}{\prod_{j=1}^{i-1}s_{j}}  \ \text{mod} \ s_{i}  \right\rfloor }_{i=1}^m $$
and $\varphi_{D}$ is the co-ordinate map given by $$ \ket{x_{i}}_{i=1}^m \mapsto \sum_{i=1}^m d_{i}x_{i} $$
whose range is $[0,\text{coSize}(L))$ and hence for any $N \ge \text{coSize(L)}$ we can factor the map $\varphi_{D}$ as $\varphi_{D}^N$ via the inclusion map. The Layout map of the layout $\phi_{L}$ is the composition of the co-ordinate map after the co-lexical inverse map. 


For programmatic use, we ignore the left half of the diagram (the colexical maps) and think of layouts as inducing map from co-ordinate space to offset space (the co-ordinate map $\varphi_D$)

when we have layouts that aren't flat, the colexical maps will help us index into it in many different ways, 
for flat layouts, you can index using the domain of the colexical_inv map (which is just a number in $[0,\text{size}(S))$) or using the domain of the co-ordinate map, which would be a co-ordinate whose index is within the span of the shape tuple. 

In [5]:
shape = (3,4,5) 
stride = (1,3,2)


@cute.jit
def make_layout_and_print(shape,stride):
  layout = cute.make_layout(shape, stride=stride)
  cute.printf(layout)
  size = cute.size(layout)
  cosize = cute.cosize(layout)
  cute.printf(size,cosize)
  for x in range(size): 
    cute.printf(layout(x)) 
  
  for x0 in range(shape[0]): 
    for x1 in range(shape[1]): 
      for x2 in range(shape[2]): 
        cute.printf(layout((x0,x1,x2)))
    

make_layout_and_print(shape,stride)

(3,4,5):(1,3,2)
60, 20
0
1
2
3
4
5
6
7
8
9
10
11
2
3
4
5
6
7
8
9
10
11
12
13
4
5
6
7
8
9
10
11
12
13
14
15
6
7
8
9
10
11
12
13
14
15
16
17
8
9
10
11
12
13
14
15
16
17
18
19
0
2
4
6
8
3
5
7
9
11
6
8
10
12
14
9
11
13
15
17
1
3
5
7
9
4
6
8
10
12
7
9
11
13
15
10
12
14
16
18
2
4
6
8
10
5
7
9
11
13
8
10
12
14
16
11
13
15
17
19


A layout $L = S:D$ is a flat layout $L^b = S^b:D^b$ and a profile $P$ (which informally is a specification of a bracketing of a flat tuple) such that $S = P(S^b), D = P(D^b)$ (we can think of the profile as a function that we can apply on the flat tuple), without going into all the mathematical details (I want this notebook to be more informal) 

But essentially a general layout is a flat layout with the same bracketing on the shape and stride tuples. 

consider the example $L = ((3,4),(5,6,7),13,9):((1,5),(2,8,9),12,4)$

the outermost shit of a layout are called modes of that layout 
for example (3,4):(1,5), (5,6,7):(2,8,9), 13:12, 9:4 are all the modes of $L$. 
Indeed, each 1d entry, taking the innermost shape and co-responding stride (by index) is called an entry of a layout. 

the length of the layout is the number of entries, for example $L$ has length 7. 
The rank of a layout is the number of modes, $L$ has rank $4$ 

Also, the application of the profile does not change the maps that are induced on the layout, the layout and flat layout share the maps, the addition of profiles just allow us to index differently into the layout,  we shall see that later. 






In [6]:
shape = ((3,4),(5,6,7),13,9)
stride = ((1,5),(2,8,9),12,4)

flat_length = 7
@cute.jit 
def make_nested_layout(shape, stride): 
  L = cute.make_layout(shape, stride=stride)
  cute.printf(f"size: {cute.size(L)}")
  cute.printf(f"co-size: {cute.cosize(L)}")
  cute.printf(f"rank: {cute.rank(L)}")
  cute.printf(f"length: {len(cute.flatten(L.shape))}")
  
  L_flat = cute.flatten(L)
  
  cute.printf("---------Printing modes-----------------")
  r = cute.rank(L)
  for x in cutlass.range_constexpr(len(shape)):   # x stays a Python int
    mode = cute.make_layout(shape[x], stride=stride[x])
    cute.printf(f"mode {x}: {mode}")
    
  cute.printf("---------Printing entries-----------------")
  for x in cutlass.range_constexpr(7): 
    entry = cute.make_layout(cute.shape(L_flat)[x], stride=L_flat.stride[x])
    cute.printf(f"entry: {x}: {entry}")


In [7]:
make_nested_layout(shape,stride)

size: 294840
co-size: 296
rank: 4
length: 7
---------Printing modes-----------------
mode 0: (3,4):(1,5)
mode 1: (5,6,7):(2,8,9)
mode 2: 13:12
mode 3: 9:4
---------Printing entries-----------------
entry: 0: 3:1
entry: 1: 4:5
entry: 2: 5:2
entry: 3: 6:8
entry: 4: 7:9
entry: 5: 13:12
entry: 6: 9:4


Indeed, then, one can view a layout as a concatenation of its modes, and thereby, cute allows us to index into a nested layout, of depth 2, (where depth 1 is a flat layout,depth 2 means 1 bracketing and flat layouts inside and so on) by either giving the full co-ordinate tuple of the mode, or giving the 1 dimensional number in the range of the size of that mode. (that is assuming the mode is not just a 1d layout of single shape and stride entry). 


that is, in the same way that we can index into a flat layout by giving a co-ordinate or single number within the size range, we can do so with modes of the nested layouts (each mode itself can be treated as its own layout) 

The way the internal calculation is done, I will demonstrate with a rank 2 layout 
suppose L = S:D can be decomposed into modes 
L1 = S1:D1 and L2 = S2:D2 ie, S = (S1,S2) and D = (D1,D2) (where S1 * S2 is flat concat, S1,S2 is tuple concat in the sense of not removing outer brackets, and L1 and L2 are flat layouts) 

#### Concatenation of Layouts: 
Two concatenate two layouts, you just concatenate their shape and stride tuples, indeed here if $L_1 = S_1: D_1$ and $L_{2} = S_{2}: D_{2}$ (where $N_1 = \text{Size}(L_{1})$, $T_{1} = \text{coSize}(L_{1})$, $N_2 = \text{Size}(L_{2})$, $T_{2} = \text{coSize}(L_{2})$) 
The induced algebra of concatenated layouts is packed into the statement "the below diagram commutes fully" the upper square and the lower triangle both commute. 
<!-- https://q.uiver.app/#q=WzAsMTQsWzAsMV0sWzIsMV0sWzQsMV0sWzAsNCwiWzAsU18xIFxcc3RhciBTXzIpIixbMjI1LDY3LDg4LDFdXSxbNCw0XSxbNCw1XSxbNCw2XSxbMywxXSxbMyw0LCJbMCxTXzEpIFxcdGltZXMgWzAsU18yKSIsWzIyNSw2Nyw4OCwxXV0sWzMsNiwiWzAsIDEgKyAoVF8xICsgVF8yIC0yKSkiLFsyMjUsNjcsODgsMV1dLFswLDBdLFswLDIsIlswLE5fMU5fMikiLFsyMjUsNjcsODgsMV1dLFszLDIsIlswLE5fMSkgXFx0aW1lcyBbMCxOXzIpICIsWzIyNSw2Nyw4OCwxXV0sWzMsNV0sWzMsOCwiWF8xIFxcc3RhciBYXzIgXFxtYXBzdG8gKFhfMSxYXzIpIiwwLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV0sWzgsOSwiXFx2YXJwaGlfe0RfMX0gKyBcXHZhcnBoaV97RF8yfSIsMix7ImNvbG91ciI6WzIyNSw2Nyw4OF19LFsyMjUsNjcsODgsMV1dLFszLDksIlxcdmFycGhpX3tEXzEgXFxzdGFyIERfMn0iLDAseyJjb2xvdXIiOlsyMjUsNjcsODhdfSxbMjI1LDY3LDg4LDFdXSxbMTEsMywiXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzEgXFxzdGFyIFNfMn0iLDAseyJjb2xvdXIiOlsyMjUsNjcsODhdfSxbMjI1LDY3LDg4LDFdXSxbMTIsOCwiXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzF9IFxcdGltZXMgXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzJ9IiwyLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV0sWzExLDEyLCJcXHRleHR7Y29sZXh9XnstMX1feyhOXzEsIE5fMil9IiwwLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV1d --> <iframe class="quiver-embed" src="https://q.uiver.app/#q=WzAsMTQsWzAsMV0sWzIsMV0sWzQsMV0sWzAsNCwiWzAsU18xIFxcc3RhciBTXzIpIixbMjI1LDY3LDg4LDFdXSxbNCw0XSxbNCw1XSxbNCw2XSxbMywxXSxbMyw0LCJbMCxTXzEpIFxcdGltZXMgWzAsU18yKSIsWzIyNSw2Nyw4OCwxXV0sWzMsNiwiWzAsIDEgKyAoVF8xICsgVF8yIC0yKSkiLFsyMjUsNjcsODgsMV1dLFswLDBdLFswLDIsIlswLE5fMU5fMikiLFsyMjUsNjcsODgsMV1dLFszLDIsIlswLE5fMSkgXFx0aW1lcyBbMCxOXzIpICIsWzIyNSw2Nyw4OCwxXV0sWzMsNV0sWzMsOCwiWF8xIFxcc3RhciBYXzIgXFxtYXBzdG8gKFhfMSxYXzIpIiwwLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV0sWzgsOSwiXFx2YXJwaGlfe0RfMX0gKyBcXHZhcnBoaV97RF8yfSIsMix7ImNvbG91ciI6WzIyNSw2Nyw4OF19LFsyMjUsNjcsODgsMV1dLFszLDksIlxcdmFycGhpX3tEXzEgXFxzdGFyIERfMn0iLDAseyJjb2xvdXIiOlsyMjUsNjcsODhdfSxbMjI1LDY3LDg4LDFdXSxbMTEsMywiXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzEgXFxzdGFyIFNfMn0iLDAseyJjb2xvdXIiOlsyMjUsNjcsODhdfSxbMjI1LDY3LDg4LDFdXSxbMTIsOCwiXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzF9IFxcdGltZXMgXFx0ZXh0e2NvbGV4fV57LTF9X3tTXzJ9IiwyLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV0sWzExLDEyLCJcXHRleHR7Y29sZXh9XnstMX1feyhOXzEsIE5fMil9IiwwLHsiY29sb3VyIjpbMjI1LDY3LDg4XX0sWzIyNSw2Nyw4OCwxXV1d&embed" width="700" height="700" style="border-radius: 8px; border: none;"></iframe>

okay using the above concat notion, and the commutative diagrams, there are a few ways to index into a layout which looks like the concatnation of two modes, which are each flat layouts: 

An example would be $L = ((3,4),(5,6,7)):((1,5),(2,8,9))$ 
its obvious that $L1 = (3,4):(1,5)$ and $L2 =(5,6,7):(2,8,9)$

the first way is to give a single number in the range $[0,N_1N_2)$ which is the size of the layout. and we can start at the top left corner of the square and take any path to the bottom right corner of the triangle. 

the second way is to give full co-ordinates, like for L we would give ((x0,x1),(x2,x3,x4)) in this case we can just flatten and use the path of bottom left of the square to the bottom right of the triangle, or bottom right of the square to the bottom right of the triangle, anything goes.

the third way is to give a singe number in the range of the first mode $[0,N_1)$ and then give co-ordinates for the other mode, which would look like (x,(x2,x3,x4)) at which point we can use $colex^{-1}_{S_1} \times id_{S_2}(x,(x2,x3,x4))$ to cast it to ((x0,x1),(x2,x3,x4)) (there are other ways too) for example we know from the diagram that we can just add the layout map of the modes, where each mode applies its layout map to the co-responding section of the input tuple. 

the fourth way is symmetrical to the third, we can give the co-ordinates of the first mode and just the 1d number of the second mode. 


In [8]:
def colex_inv(shape, x): 
  co_ordinate_list = [0]*len(shape)
  colex_stride = [1]*len(shape)
  for i in range(1,len(shape)): 
    colex_stride[i] = colex_stride[i-1]*shape[i-1]
    
  for i in range(len(shape)): 
    co_ordinate_list[i] = (x//colex_stride[i]) % shape[i]
    
  return tuple(co_ordinate_list)

shape_1 = (3,4)
stride_1 = (1,5)
shape_2 = (5,6,7)
stride_2 = (2,8,9)
shape = (shape_1,shape_2)
stride = (stride_1, stride_2)
size_1 = 12

@cute.jit
def indexing_depth_2_rank_2_nested_layout(shape,stride,shape_1,stride_1,shape_2,stride_2): 
  L_1 = cute.make_layout(shape_1, stride=stride_1)
  L_2 = cute.make_layout(shape_2, stride=stride_2) 
  L = cute.make_layout(shape,stride=stride)
  #we will do the indexing of mode1 as 1d number, and mode2 and 3d number (full co-ordinates)
  for x in cutlass.range_constexpr(size_1): 
    for x2 in range(shape_2[0]): 
      for x3 in range(shape_2[1]): 
        for x4 in range(shape_2[2]):
          layout_applied = L((x,(x2,x3,x4)))
          layout_summed = L_1(x) + L_2((x2,x3,x4))
          cute.printf(f"L({x,(x2,x3,x4)}) = L_1({x}) + L_2({(x2,x3,x4)} eqivalently {layout_applied} = {layout_summed})")
  



In [9]:
indexing_depth_2_rank_2_nested_layout(shape,stride,shape_1,stride_1,shape_2,stride_2)

L((0,(0,0,0))) = L_1(0) + L_2((0,0,0) eqivalently 0 = 0)
L((0,(0,0,1))) = L_1(0) + L_2((0,0,1) eqivalently 9 = 9)
L((0,(0,0,2))) = L_1(0) + L_2((0,0,2) eqivalently 18 = 18)
L((0,(0,0,3))) = L_1(0) + L_2((0,0,3) eqivalently 27 = 27)
L((0,(0,0,4))) = L_1(0) + L_2((0,0,4) eqivalently 36 = 36)
L((0,(0,0,5))) = L_1(0) + L_2((0,0,5) eqivalently 45 = 45)
L((0,(0,0,6))) = L_1(0) + L_2((0,0,6) eqivalently 54 = 54)
L((0,(0,1,0))) = L_1(0) + L_2((0,1,0) eqivalently 8 = 8)
L((0,(0,1,1))) = L_1(0) + L_2((0,1,1) eqivalently 17 = 17)
L((0,(0,1,2))) = L_1(0) + L_2((0,1,2) eqivalently 26 = 26)
L((0,(0,1,3))) = L_1(0) + L_2((0,1,3) eqivalently 35 = 35)
L((0,(0,1,4))) = L_1(0) + L_2((0,1,4) eqivalently 44 = 44)
L((0,(0,1,5))) = L_1(0) + L_2((0,1,5) eqivalently 53 = 53)
L((0,(0,1,6))) = L_1(0) + L_2((0,1,6) eqivalently 62 = 62)
L((0,(0,2,0))) = L_1(0) + L_2((0,2,0) eqivalently 16 = 16)
L((0,(0,2,1))) = L_1(0) + L_2((0,2,1) eqivalently 25 = 25)
L((0,(0,2,2))) = L_1(0) + L_2((0,2,2) eqivalently 34 = 34)
L((

Indeed, this can be used recursively, as long as you know how to index depth N-1 layouts, you can index into depth N layouts by indexing using the depth N-1 logic on each mode and then summing up the results. 
